In [1]:
%load_ext autoreload
%autoreload 2
import sys, os

sys.path.insert(0, os.path.abspath("C:/Users/Voror/Projects/Personal/sep"))

from SIDER_dataset.libraries.utils import get_clus_path
from sklearn.model_selection import KFold
from SIDER_dataset.libraries.XofN_library import *
from SIDER_dataset.libraries.PCT_library import run_PCT


In [2]:
# Set ADR to predict and scoring
clus_path = get_clus_path()
paths = get_dataset_paths()
print(len(paths), "datasets")
len(paths)

TEST = False
if TEST:
    paths = [paths[0]]

18 datasets


In [3]:
paths = [path for path in paths if "ten_mid" in path["dataset_name"]]
len(paths)

3

In [4]:
k = 10
random_state = 42
performances = []
training_algorithm = "PCT"
eval_criteria = ["averageAUROC", "HammingLoss", "SubsetAccuracy", "RankingLoss", "MacroPrecision", "MacroRecall",
                 "MacroFOne"]

cv_results = []
for idx, path in enumerate(paths, start=1):
    run_config = f"\n--- Running with label:'{path["dataset_path"]}' training_algorithm:'{training_algorithm}' ---"
    print(run_config)
    # Load dataset
    current_df = pd.read_csv(path["dataset_path"])
    if TEST:
        features = get_features(current_df, path["label_set"])
        current_df = current_df[features[:10] + path["label_set"]]
    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
    for fold, (train_idx, test_idx) in enumerate(kf.split(current_df), start=1):
        print(f"\nFold {fold}/{k} ({path["dataset_name"]} {idx}/{len(paths)})")
        train_dataset = current_df.iloc[train_idx]
        test_dataset = current_df.iloc[test_idx]

        train_dataset.to_csv(f"XofN_none/tmp/train_dataset.csv", index=False)
        test_dataset.to_csv(f"XofN_none/tmp/test_dataset.csv", index=False)
        os.makedirs("XofN_none/tmp", exist_ok=True)
        os.makedirs("XofN_none/tmp", exist_ok=True)

        original_res, pruned_res, training_time = run_PCT(clus_path,
                                                          "XofN_none/tmp/train_dataset.csv",
                                                          path["label_set"],
                                                          eval_criteria,
                                                          test_dataset_path=f"XofN_none/tmp/test_dataset.csv")
        pruned_performance = get_fold_results(pruned_res, eval_criteria, True, fold, True, [[]], 0, training_time,
                                              path["dataset_name"])
        performances.append(pruned_performance)
        performance = get_fold_results(original_res, eval_criteria, False, fold, True, [[]], 0, training_time,
                                       path["dataset_name"])
        performances.append(performance)
    final_perf_df = pd.DataFrame(performances)
    averages = final_perf_df.groupby(["pruning", 'dataset'])[
        eval_criteria + ["nodes", "leaves", 'training_time']].mean().reset_index()
    print(averages)
    cv_results.append(averages)
    performances = []
# laptop - 7m
# desktop - 5m


--- Running with label:'C:\Users\Voror\Projects\Personal/sep/SIDER_dataset/datasets/clean_multi_label_datasets/CPI+fingerprint_ten_mid.csv' training_algorithm:'PCT' ---

Fold 1/10 (CPI+fingerprint_ten_mid 1/3)
pruning: True, include_original_features: with_org, averageAUROC: 0.5912864, HammingLoss: 0.37482, SubsetAccuracy: 0.10791, RankingLoss: 0.34238, MacroPrecision: 0.46868, MacroRecall: 0.32403, MacroFOne: 0.37642, 
pruning: False, include_original_features: with_org, averageAUROC: 0.5956056, HammingLoss: 0.38993, SubsetAccuracy: 0.043165, RankingLoss: 0.34976, MacroPrecision: 0.4635, MacroRecall: 0.51521, MacroFOne: 0.48525, 

Fold 2/10 (CPI+fingerprint_ten_mid 1/3)
pruning: True, include_original_features: with_org, averageAUROC: 0.6710237, HammingLoss: 0.31727, SubsetAccuracy: 0.13669, RankingLoss: 0.30883, MacroPrecision: 0.54313, MacroRecall: 0.43216, MacroFOne: 0.47998, 
pruning: False, include_original_features: with_org, averageAUROC: 0.6482778, HammingLoss: 0.36835, Subse

In [5]:
# All results
save_path = "XofN_none/"
final_grouped_res = pd.DataFrame()
for df in cv_results:
    final_grouped_res = pd.concat([final_grouped_res, df])
final_grouped_res.sort_values(by='dataset', ascending=False, inplace=True)
final_grouped_res = final_grouped_res.reset_index(drop=True)
final_grouped_res.to_csv(save_path + "all_results.csv")
final_grouped_res

,pruning,dataset,averageAUROC,HammingLoss,SubsetAccuracy,RankingLoss,MacroPrecision,MacroRecall,MacroFOne,nodes,leaves,training_time
0,False,fingerprint_ten_mid,0.576249,0.418847,0.040418,0.348240,0.418015,0.504722,0.453971,992.4,496.7,1.103360
1,True,fingerprint_ten_mid,0.580143,0.349812,0.109837,0.349611,0.504373,0.248164,0.323269,85.6,43.3,1.103360
2,False,CPI_ten_mid,0.577627,0.414529,0.049656,0.349587,0.441284,0.479544,0.454425,762.6,381.8,2.766194
3,True,CPI_ten_mid,0.571223,0.354736,0.106642,0.372242,0.532257,0.273976,0.352215,68.8,34.9,2.766194
4,False,CPI+fingerprint_ten_mid,0.593290,0.395976,0.049771,0.346985,0.438480,0.506388,0.467329,1034.2,517.6,3.097539
5,True,CPI+fingerprint_ten_mid,0.597710,0.344419,0.131346,0.343164,0.504136,0.279675,0.353659,102.4,51.7,3.097539


In [6]:
# Table ready (with pruning, rounded, compact)
save_path = "XofN_none/"
rounded_final_grouped_res = pd.read_csv(save_path + "all_results.csv", index_col=0)
rounded_final_grouped_res = rounded_final_grouped_res[rounded_final_grouped_res["pruning"] == True]
for measure in eval_criteria:
    rounded_final_grouped_res[measure] = rounded_final_grouped_res[measure].round(3)

rounded_final_grouped_res["Nodes; Leaves"] = rounded_final_grouped_res["nodes"].round(1).astype(str) + "; " + \
                                             rounded_final_grouped_res["leaves"].round(1).astype(str)

rounded_final_grouped_res["PCT tr. time (s)"] = rounded_final_grouped_res["training_time"].round(
    1).astype(str)

rounded_final_grouped_res = rounded_final_grouped_res.drop(columns=["pruning", "nodes", "leaves", "training_time"])
rounded_final_grouped_res.to_csv(save_path + "table_results.csv")
rounded_final_grouped_res

,dataset,averageAUROC,HammingLoss,SubsetAccuracy,RankingLoss,MacroPrecision,MacroRecall,MacroFOne,Nodes; Leaves,PCT tr. time (s)
1,fingerprint_ten_mid,0.580,0.350,0.110,0.350,0.504,0.248,0.323,85.6; 43.3,1.1
3,CPI_ten_mid,0.571,0.355,0.107,0.372,0.532,0.274,0.352,68.8; 34.9,2.8
5,CPI+fingerprint_ten_mid,0.598,0.344,0.131,0.343,0.504,0.280,0.354,102.4; 51.7,3.1
